In [ ]:
from torch_uncertainty.metrics import AURC

import sys
sys.path.append("../scripts")

import numpy as np
from pathlib import Path
import torch
from scipy.special import log_softmax

from callm.data.classification import DATASETS


%load_ext autoreload
%autoreload 2


In [ ]:
class AUROCMetric(AURC):

    def update(self, confidences, correctness):
        self.scores.append(confidences.float())
        self.errors.append((1 - correctness).float())


aurc = AUROCMetric()
for dataset_model in DATASETS:
    data_path = Path("../scores/classification") / dataset_model
    logprobs = torch.from_numpy(log_softmax(np.load(data_path / "scores.npy"), axis=1)).float()
    labels = torch.from_numpy(np.load(data_path / "targets.npy").reshape(-1))
    confidences = torch.exp(logprobs).max(dim=1).values
    correctness = (logprobs.argmax(dim=1) == labels).float()
    aurc.update(confidences, correctness)
    aurc_score = aurc.compute()
    aurc.reset()
    # shifted_confidences = confidences + 0.5
    shifted_confidences = torch.maximum(torch.tensor(1.0), confidences + 0.8)
    aurc.update(shifted_confidences, correctness)
    shifted_aurc_score = aurc.compute()
    aurc.reset()
    print(f"{dataset_model} - AURC: {aurc_score:.4f}, Shifted AURC: {shifted_aurc_score:.4f}, diff: {(shifted_aurc_score - aurc_score) if (shifted_aurc_score - aurc_score) > 1e-3 else '< 1e-3'}")
